In [1]:
import os
import pandas as pd
import kagglehub

path = kagglehub.dataset_download("mrwellsdavid/unsw-nb15")
train = pd.read_csv(os.path.join(path, "UNSW_NB15_testing-set.csv"))   # 175 341 lignes
test = pd.read_csv(os.path.join(path, "UNSW_NB15_training-set.csv"))   # 82 332 lignes
print(train.shape, test.shape)

(175341, 45) (82332, 45)


In [2]:
cat_cols = ["proto", "service", "state"]
drop_cols = ["id", "label", "attack_cat"]
num_cols = [c for c in train.columns if c not in cat_cols + drop_cols]

X_train, y_train = train.drop(columns=drop_cols), train["label"]
X_test, y_test = test.drop(columns=drop_cols), test["label"]

print(len(num_cols), "colonnes numériques,", len(cat_cols), "colonnes catégorielles")

39 colonnes numériques, 3 colonnes catégorielles


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist",
                          min_frequency=500, sparse_output=False), cat_cols),
])

X_train_p = preprocessor.fit_transform(X_train)   # apprend sur le train
X_test_p = preprocessor.transform(X_test)         # applique seulement au test

print("Avant :", X_train.shape, "| Après :", X_train_p.shape)

Avant : (175341, 42) | Après : (175341, 60)


In [4]:
noms = preprocessor.get_feature_names_out()
print([n for n in noms if n.startswith("cat__")])

['cat__proto_arp', 'cat__proto_ospf', 'cat__proto_sctp', 'cat__proto_tcp', 'cat__proto_udp', 'cat__proto_unas', 'cat__proto_infrequent_sklearn', 'cat__service_-', 'cat__service_dns', 'cat__service_ftp', 'cat__service_ftp-data', 'cat__service_http', 'cat__service_pop3', 'cat__service_smtp', 'cat__service_ssh', 'cat__service_infrequent_sklearn', 'cat__state_CON', 'cat__state_FIN', 'cat__state_INT', 'cat__state_REQ', 'cat__state_infrequent_sklearn']


In [5]:
import joblib

os.makedirs("../models", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

joblib.dump(preprocessor, "../models/preprocessor.joblib")
joblib.dump((X_train_p, y_train, X_test_p, y_test),
            "../data/processed/data_processed.joblib", compress=3)

print("Sauvegarde terminée")

Sauvegarde terminée


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CAT_COLS = ["proto", "service", "state"]
DROP_COLS = ["id", "label", "attack_cat"]

def build_preprocessor(num_cols):
    return ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist",
                              min_frequency=500, sparse_output=False), CAT_COLS),
    ])